# WorldPop — multi-country mosaic (live)

A bbox over the Rwanda–Burundi border spans several countries. WorldPop is per-country, so earthlens downloads each country's raster and mosaics + crops to the AOI. Uses 1 km (`wpic1km`) to keep it small.

## What this notebook does

1. **Download** — build a WorldPop request for a cross-border AOI, see which countries it spans, and download + mosaic + crop each country's 1 km population raster to the bbox.
2. **Inspect** — read the mosaicked GeoTIFF back with pyramids' `Dataset` to confirm its CRS and shape.

> WorldPop is open (CC-BY-4.0) and needs no credentials.

## Setup

The imports: `tempfile` for a throwaway download directory, `earthlens` for the unified `EarthLens` entry point, and pyramids' `Dataset` for reading the result back. We also make a temporary directory to hold the downloaded rasters.

In [ ]:
import tempfile

from pyramids.dataset import Dataset

from earthlens.core import EarthLens

download_dir = tempfile.mkdtemp()

## 1 · Download the multi-country mosaic

WorldPop serves population as one raster per country, so we point a single AOI at the Rwanda–Burundi border and let earthlens fetch every country it touches.

### Build the request

We ask for the `pop` (population count) variable for 2020 at 1 km resolution (`wpic1km`) over a bbox straddling the border. The request is built first, on its own, so it's easy to read and tweak.

In [ ]:
backend = EarthLens(
    data_source="worldpop",
    variables=["pop"],
    start="2020",
    end="2020",
    fmt="%Y",
    aoi=[29.0, -2.9, 30.0, -2.3],
    path=download_dir,
    resolution="1km",
)

### Which countries does the AOI span?

`_iso3s` lists the ISO3 country codes the bounding box overlaps — these are the per-country rasters earthlens will fetch and mosaic together.

In [ ]:
print("countries in AOI:", backend._iso3s)

### Download, mosaic and crop

`download()` fetches each country's raster, mosaics them, and crops the result to the AOI, returning the list of written file paths.

In [ ]:
paths = backend.download(progress_bar=False)
print("written:", [p.name for p in paths])

## 2 · Inspect the result

Reading the mosaicked GeoTIFF back with pyramids' `Dataset` confirms the output CRS and array shape. We grab the first written path, then read it.

In [ ]:
mosaic_path = paths[0]
ds = Dataset.read_file(mosaic_path)
print("mosaicked + cropped EPSG:", ds.epsg, "shape:", ds.shape)